# Lab 2: Automating Dataset Extension

**Course:** ML Operations  
**Objective:** Design a configuration-driven system for selecting data batches, combining them into dynamic train/val sets while keeping the test set static, and demonstrating how different data subsets affect model performance.

## 2.1 Setup and Imports

In [1]:
import os
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(repo_root)

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(level=logging.INFO)

print(f'Working directory: {Path.cwd()}')

Working directory: /home/d1pg1/ml_engineering_labs


In [2]:
from typing import Any, Dict, List

import torch
import torch.nn as nn
import torch.optim as optim
import yaml

from src.data.batch_dataset import create_batch_splits, divide_into_batches, select_batches
from src.data.dataset import (
    create_data_loader,
    eval_transform,
    load_labels,
    train_test_split,
    train_transform,
)
from src.data.download import download_and_extract
from src.models.cnn import CifarCNN
from src.training.evaluate import test_model
from src.training.trainer import train_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 2.2 Part 1: Dataset Configuration and Management

### 2.2.1 Load Configuration

All pipeline parameters are controlled via `configs/lab02.yaml`. This ensures reproducibility and makes it trivial to switch batch configurations without touching code.

In [3]:
with open('configs/lab02.yaml', 'r') as f:
    config: Dict[str, Any] = yaml.safe_load(f)

print('Loaded config:')
print(yaml.dump(config, default_flow_style=False))

Loaded config:
batches:
  seed: 42
  total: 5
  train: 3
  val: 1
data:
  cifar_url: https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
  data_dir: data
  n_classes: 10
  random_state: 42
  test_size: 0.2
  val_size: 0.2
output:
  save_path: outputs/lab02_model.pth
training:
  batch_size: 128
  lr: 0.001
  num_epochs: 10
  num_workers: 2



### 2.2.2 Data Download and Ingestion

Download CIFAR-10 if not already present, then unpack all 60,000 images to disk as JPEGs.

In [4]:
data_cfg: Dict[str, Any] = config['data']

data_dir = download_and_extract(data_cfg['cifar_url'], data_cfg['data_dir'])
cifar_batches_dir = str(Path(data_dir) / 'cifar-10-batches-py')

# Load all 60,000 images into a single DataFrame
full_df = load_labels(cifar_batches_dir)
print(f'Full dataset size: {len(full_df)} images')

INFO:root:Downloading 'cifar-10-python.tar.gz' from 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'...


INFO:root:Download successful. File saved to: 'data/cifar-10-python.tar.gz'


INFO:root:Loading batch: data_batch_1


INFO:root:Loading batch: data_batch_2


INFO:root:Loading batch: data_batch_3


INFO:root:Loading batch: data_batch_4


INFO:root:Loading batch: data_batch_5


INFO:root:Loading batch: test_batch


INFO:root:Loaded 60000 images from CIFAR-10 batches.


Full dataset size: 60000 images


### 2.2.3 Static Test Set + Batch Division

The test set is created **once** with a fixed random seed and never changes across experiments — this guarantees a fair comparison between batch configurations.

The remaining train+val pool is then divided into `batches.total` equal chunks. Each experiment selects a different number of chunks for training, while validation always uses the same single chunk.

In [5]:
batch_cfg: Dict[str, Any] = config['batches']

# Carve out the static test set first — seed is fixed and never changes
train_val_pool, test_df = train_test_split(
    full_df,
    test_size=data_cfg['test_size'],
    random_state=data_cfg['random_state'],
)

# Divide the remaining pool into N batches with a fixed seed
all_batches = divide_into_batches(
    train_val_pool,
    n_batches=batch_cfg['total'],
    random_state=batch_cfg['seed'],
)

print(f'Static test set:  {len(test_df):,} images')
print(f'Train+val pool:   {len(train_val_pool):,} images')
print(f'Batches created:  {len(all_batches)} batches')
print(f'Samples per batch: ~{len(all_batches[0]):,}')

INFO:root:Divided 48000 samples into 5 batches (~9600 samples each).


Static test set:  12,000 images
Train+val pool:   48,000 images
Batches created:  5 batches
Samples per batch: ~9,600


## 2.3 Part 2: Model Training and Evaluation

Three experiments vary the number of training batches while keeping the validation batch and test set constant.

| Experiment | Train batches | Train samples | Val batches | Val samples | Test samples |
|-----------|:---:|---:|:---:|---:|---:|
| A | 1 of 5 | ~9,600 | 1 of 5 | ~9,600 | 12,000 |
| B | 3 of 5 | ~28,800 | 1 of 5 | ~9,600 | 12,000 |
| C | 4 of 5 | ~38,400 | 1 of 5 | ~9,600 | 12,000 |

In [6]:
def run_experiment(
    name: str,
    train_batch_ids: List[int],
    val_batch_id: int,
    all_batches: list,
    test_df,
    config: Dict[str, Any],
    device: torch.device,
) -> Dict[str, Any]:
    """Train and evaluate the CNN with the specified batch selection."""
    logging.info(f'=== Experiment {name}: train_batches={train_batch_ids}, val_batch={val_batch_id} ===')

    train_cfg = config['training']
    out_cfg = config['output']

    train_df = select_batches(all_batches, train_batch_ids)
    val_df = select_batches(all_batches, [val_batch_id])

    train_loader = create_data_loader(
        '', train_df, {**train_cfg, 'transform': train_transform}
    )
    val_loader = create_data_loader(
        '', val_df, {**train_cfg, 'transform': eval_transform}
    )
    test_loader = create_data_loader(
        '', test_df, {**train_cfg, 'transform': eval_transform}
    )

    model = CifarCNN(n_classes=config['data']['n_classes']).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=train_cfg['lr'])

    save_path = Path(out_cfg['save_path']).with_stem(f'lab02_{name}')
    save_path.parent.mkdir(parents=True, exist_ok=True)

    best_path = train_model(
        model, train_loader, val_loader, loss_fn, optimizer,
        num_epochs=train_cfg['num_epochs'],
        device=device,
        save_path=save_path,
    )

    model.load_state_dict(torch.load(best_path, map_location=device))
    metrics = test_model(model, test_loader, loss_fn, device)

    return {
        'experiment': name,
        'train_samples': len(train_df),
        'val_samples': len(val_df),
        'test_samples': len(test_df),
        **metrics,
    }

### Experiment A — 1 training batch (~9,600 samples)

In [7]:
result_a = run_experiment(
    name='A',
    train_batch_ids=[0],
    val_batch_id=4,
    all_batches=all_batches,
    test_df=test_df,
    config=config,
    device=device,
)
print(result_a)

INFO:root:=== Experiment A: train_batches=[0], val_batch=4 ===


INFO:root:Selected batches [0] → 9600 samples.


INFO:root:Selected batches [4] → 9600 samples.


INFO:root:Epoch 1/10, Loss: 1.7562


INFO:root:Epoch 1/10, Validation Loss: 1.7074


...

INFO:root:Epoch 10/10, Validation Loss: 1.0437


INFO:root:Best model saved with validation loss: 1.0437


INFO:root:Training complete.


INFO:root:Test Results — Loss: 1.0386 | Accuracy: 0.6321 | Precision: 0.6307 | Recall: 0.6321 | F1: 0.6288


{'experiment': 'A', 'train_samples': 9600, 'val_samples': 9600, 'test_samples': 12000, 'loss': 1.0386054858248284, 'accuracy': 0.6320833333333333, 'precision': 0.6307007584426014, 'recall': 0.6320833333333333, 'f1': 0.6288324623182369}


### Experiment B — 3 training batches (~28,800 samples)

In [8]:
result_b = run_experiment(
    name='B',
    train_batch_ids=[0, 1, 2],
    val_batch_id=4,
    all_batches=all_batches,
    test_df=test_df,
    config=config,
    device=device,
)
print(result_b)

INFO:root:=== Experiment B: train_batches=[0, 1, 2], val_batch=4 ===


INFO:root:Selected batches [0, 1, 2] → 28800 samples.


INFO:root:Selected batches [4] → 9600 samples.


INFO:root:Epoch 1/10, Loss: 1.3247


INFO:root:Epoch 1/10, Validation Loss: 1.4692


...

INFO:root:Epoch 10/10, Loss: 1.1293


INFO:root:Epoch 10/10, Validation Loss: 0.9242


INFO:root:Training complete.


INFO:root:Test Results — Loss: 0.8885 | Accuracy: 0.6844 | Precision: 0.6902 | Recall: 0.6844 | F1: 0.6786


{'experiment': 'B', 'train_samples': 28800, 'val_samples': 9600, 'test_samples': 12000, 'loss': 0.8884774535260302, 'accuracy': 0.6844166666666667, 'precision': 0.6902003638864418, 'recall': 0.6844166666666667, 'f1': 0.6785574277447425}


### Experiment C — 4 training batches (~38,400 samples)

In [9]:
result_c = run_experiment(
    name='C',
    train_batch_ids=[0, 1, 2, 3],
    val_batch_id=4,
    all_batches=all_batches,
    test_df=test_df,
    config=config,
    device=device,
)
print(result_c)

INFO:root:=== Experiment C: train_batches=[0, 1, 2, 3], val_batch=4 ===


INFO:root:Selected batches [0, 1, 2, 3] → 38400 samples.


INFO:root:Selected batches [4] → 9600 samples.


INFO:root:Epoch 1/10, Loss: 1.4086


INFO:root:Epoch 1/10, Validation Loss: 1.3443


...

INFO:root:Epoch 10/10, Validation Loss: 0.8132


INFO:root:Best model saved with validation loss: 0.8132


INFO:root:Training complete.


INFO:root:Test Results — Loss: 0.8260 | Accuracy: 0.7103 | Precision: 0.7177 | Recall: 0.7103 | F1: 0.7067


{'experiment': 'C', 'train_samples': 38400, 'val_samples': 9600, 'test_samples': 12000, 'loss': 0.8260246787933593, 'accuracy': 0.71025, 'precision': 0.7177405604964587, 'recall': 0.71025, 'f1': 0.7066795274875723}


### Results Comparison

In [10]:
import pandas as pd

results_df = pd.DataFrame([result_a, result_b, result_c])
results_df = results_df.set_index('experiment')
results_df[['train_samples', 'val_samples', 'loss', 'accuracy', 'precision', 'recall', 'f1']].round(4)

,train_samples,val_samples,loss,accuracy,precision,recall,f1
experiment,,,,,,,
A,9600,9600,1.0386,0.6321,0.6307,0.6321,0.6288
B,28800,9600,0.8885,0.6844,0.6902,0.6844,0.6786
C,38400,9600,0.8260,0.7103,0.7177,0.7103,0.7067


In [11]:
print('=== Final Results on Static Test Set ===')
print(f'{"Experiment":<12} {"Train Samples":<16} {"Accuracy":<12} {"Precision":<12} {"Recall":<12} {"F1":<10}')
print('-' * 74)
for r in [result_a, result_b, result_c]:
    print(
        f'{r["experiment"]:<12} {r["train_samples"]:<16,} '
        f'{r["accuracy"]:<12.4f} {r["precision"]:<12.4f} '
        f'{r["recall"]:<12.4f} {r["f1"]:<10.4f}'
    )

=== Final Results on Static Test Set ===
Experiment   Train Samples    Accuracy     Precision    Recall       F1        
--------------------------------------------------------------------------
A            9,600            0.6321       0.6307       0.6321       0.6288    
B            28,800           0.6844       0.6902       0.6844       0.6786    
C            38,400           0.7103       0.7177       0.7103       0.7067    


## 2.4 Part 3: Code Management and Best Practices

- **Configuration:** All parameters controlled via `configs/lab02.yaml` — no hardcoded values.
- **Logging:** Python `logging` module used throughout; no `print()` in source modules.
- **Code quality:** `mypy`, `ruff`, `black`, `isort` enforced via `pyproject.toml` dev-dependencies.
- **Dependency management:** Poetry + `requirements.txt` for reproducible environment.
- **Version control:** Git with meaningful commit messages.

## 2.5 Part 4: Lab Report

See `docs/lab_report_02.md` for the written report.